## Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. Langchain supports multiple shcema types and methods for enforcing structured output

## Pydantic

Pydantic models provide the richest feature set with field validation, descriptions and nested structures

In [3]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]= os.getenv("GROQ_API_KEY")

model= init_chat_model("qwen/qwen3.8-27b",model_provider="groq")

In [4]:
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, client=<groq.resources.chat.completions.Completions object at 0x10e23b620>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10e44c1a0>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [5]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str= Field(description= "This title of the movie")
    year:int= Field(description=" This year the movie was released")
    director:str= Field(description= "The director of the movie")
    rating:float= Field(description="The movies ratings out of 10")


In [7]:
model_with_structure= model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, client=<groq.resources.chat.completions.Completions object at 0x10e23b620>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10e44c1a0>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'This title of the movie', 'type': 'string'}, 'year': {'description': ' This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies ratings out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'function': {'name': 'Movie', 'descr

In [9]:
model.invoke("Provide details about the movie Inception")

AIMessage(content='**Inception** is a 2010 science fiction action thriller film written and directed by **Christopher Nolan**. It is widely regarded as one of the most complex and influential films of the 21st century, blending heist movie tropes with philosophy, psychology, and quantum physics concepts.\n\n### **Basic Information**\n- **Release Year:** 2010\n- **Director/Writer:** Christopher Nolan\n- **Producers:** Emma Thomas, Christopher Nolan, Jonathan Golember\n- **Studio:** Legendary Pictures, Syncopy Inc.\n- **Runtime:** 148 minutes\n- **Rating:** PG-13 (for sci-fi action, peril, some language, and brief nudity)\n\n### **Main Cast**\n- **Leonardo DiCaprio** as **Dom Cobb** – A skilled thief who steals secrets from deep within the subconscious during the dream state.\n- **Joseph Gordon-Levitt** as **Arthur** – Cobb’s “point man,” the second-in-command who handles logistics and planning.\n- **Elliot Page (formerly Ellen Page)** as **Ariadne** – A brilliant architecture student wh

In [8]:
model_with_structure.invoke("Provide details about the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

## Message Output alongside parsed structure

In [10]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str= Field(description= "This title of the movie")
    year:int= Field(description=" This year the movie was released")
    director:str= Field(description= "The director of the movie")
    rating:float= Field(description="The movies ratings out of 10")

response= model_with_structure = model.with_structured_output(Movie, include_raw= True)
response

{
  raw: _ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, client=<groq.resources.chat.completions.Completions object at 0x10e23b620>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10e44c1a0>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'This title of the movie', 'type': 'string'}, 'year': {'description': ' This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies ratings out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'function': {'name': 'Movie

## Nested Structure

In [13]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year: int
    cast: list[Actor]
    genres:list[str]
    budget:float | None= Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response= model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Cillian Murphy', role='Robert Fischer')], genres=['Action', 'Science Fiction', 'Thriller'], budget=160.0)

## TypedDict

TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need run time validation

In [15]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details """

    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int,...,"The year the movie was released"]
    director: Annotated[str,..., "The director of the movie"]
    rating: Annotated[float,...,"The Movie's rating out of 10"]

model_withtypedict= model.with_structured_output(MovieDict)
response= model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [16]:
from pydantic import BaseModel, Field

class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year: int
    cast: list[Actor]
    genres:list[str]
    budget:float | None= Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response= model_with_structure.invoke("Provide details about the movie Inception")
response

{'budget': 160000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Elliot Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Eames'},
  {'name': 'Ken Watanabe', 'role': 'Saito'},
  {'name': 'Cillian Murphy', 'role': 'Robert Fischer'},
  {'name': 'Marion Cotillard', 'role': 'Mal Cobb'}],
 'genres': ['Action', 'Sci-Fi', 'Thriller'],
 'title': 'Inception',
 'year': 2010}

In [20]:
model.profile

## Data Classes

A data class is a class typically containing mainly data, although there aren't really any restrictions. you create it using the @dataclass decorator

In [22]:
import os
os.environ["OPENAI_API_KEY"]= os.getenv("OPENAI_API_KEY")

In [24]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact Information for a person. """
    name: str= Field(description="The name of the person")
    email:str= Field(description="The email address of the person")
    phone:str= Field(description="The phone number of the person")

agent= create_agent(model="gpt-5",
                    response_format= ContactInfo #Auto- Selects ProviderStrategy
                    )

result= agent.invoke({"messages":[{"role":"user","content":"Extract contact info from: John DOe, john@example.com,(555) 123-4567"}]}) 

result
#print(result["structured_response"])

#ContactInfo(name='John Doe',email=john@example.com, phone= '(555) 123- 4567')

OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [25]:
result['structured_response']

NameError: name 'result' is not defined

In [27]:
## Typedict

from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact Information for a person. """
    name: str= Field(description="The name of the person")
    email:str= Field(description="The email address of the person")
    phone:str= Field(description="The phone number of the person")

agent= create_agent(model="gpt-5",
                    response_format= ContactInfo #Auto- Selects ProviderStrategy
                    )

result= agent.invoke({"messages":[{"role":"user","content":"Extract contact info from: John DOe, john@example.com,(555) 123-4567"}]}) 
result['structured_response']

OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [30]:
#### Data Class


from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """ Contact Information for a person. """
    name: str # The name of the person
    email:str # The email address of the person
    phone: str #The phone number of a person

agent=  create_agent(
        model= "gpt-5", 
        response_format = ContactInfo  #Auto- Selects ProviderStrategy

)

result= agent.invoke({"messages":[{"role":"user","content":"Extract contact info from: John Doe, john@example.com, (555) 123- 4567"}]})

result["structured_response"]


OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}